# RTF Document Parser with OpenAI and BigQuery Integration

This notebook processes Ukrainian court documents in RTF format:
1. Downloads RTF files from Google Cloud Storage
2. Converts RTF to text
3. Extracts structured information using OpenAI API
4. Stores results in BigQuery

## Pipeline Overview
- Read document list from `documents.csv`
- Filter documents (customizable)
- Process in batches: Download → Parse → Extract → Store

## 1. Environment Setup

In [ ]:
import json
import subprocess
import re
import hashlib
from pathlib import Path
from datetime import datetime
from typing import Any, List, Dict, Optional
import tempfile
import os

import pandas as pd
from google.cloud import storage
from google.cloud import bigquery
from openai import OpenAI

storage_client = storage.Client()
bq_client = bigquery.Client()
openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

TEMP_DIR = Path(tempfile.gettempdir()) / "legal_doc_parser"
TEMP_DIR.mkdir(exist_ok=True)

print("✓ Environment initialized")
print(f"✓ Temp directory: {TEMP_DIR}")

### Configuration

In [ ]:
GCS_BUCKET_NAME = "your-bucket-name"
BQ_DATASET = "your_dataset"
BQ_TABLE = "parsed_documents"
OPENAI_MODEL = "gpt-4o-2024-08-06"
BATCH_SIZE = 10

## 2. RTF Document Parsing Functions

In [ ]:
def download_rtf_from_gcs(document_id: str, bucket_name: str) -> Path:
    """
    Download RTF file from Google Cloud Storage to temporary folder.
    
    Args:
        document_id: The document ID (also used as blob name in GCS)
        bucket_name: GCS bucket name
        
    Returns:
        Path to the downloaded RTF file
    """
    bucket = storage_client.bucket(bucket_name)
    blob_name = f"{document_id}.rtf"
    blob = bucket.blob(blob_name)
    
    local_path = TEMP_DIR / blob_name
    
    try:
        blob.download_to_filename(str(local_path))
        print(f"✓ Downloaded: {blob_name}")
        return local_path
    except Exception as e:
        raise RuntimeError(f"Failed to download {blob_name}: {e}")


def parse_rtf_to_txt(rtf_path: Path, output_path: Optional[Path] = None) -> str:
    """
    Parse RTF file and convert to plain text.
    
    Args:
        rtf_path: Path to the RTF file
        output_path: Optional path to save the TXT file
        
    Returns:
        The text content as string
        
    Note:
        This function uses a placeholder conversion logic.
        Update the conversion method as needed.
    """
    # TODO: Update this conversion logic
    # Currently using pandoc as placeholder
    cmd = ["pandoc", "-f", "rtf", "-t", "plain", "--wrap=none", str(rtf_path)]
    
    try:
        result = subprocess.run(
            cmd, 
            capture_output=True, 
            text=True, 
            timeout=60
        )
        
        if result.returncode != 0:
            raise RuntimeError(f"Conversion failed: {result.stderr[:500]}")
        
        text_content = result.stdout
        
        if output_path:
            output_path.write_text(text_content, encoding="utf-8")
            print(f"✓ Saved TXT: {output_path.name}")
        
        return text_content
    
    except FileNotFoundError:
        raise RuntimeError(
            "pandoc not found. Install: brew install pandoc (macOS) or apt-get install pandoc (Linux)"
        )
    except subprocess.TimeoutExpired:
        raise RuntimeError(f"Conversion timed out for {rtf_path.name}")

## 3. Information Extraction with OpenAI

### Prompt Configuration

In [ ]:
SYSTEM_PROMPT = """
TODO: Add your system prompt here.
This prompt will define the role and behavior of the AI.
"""

USER_PROMPT_TEMPLATE = """
TODO: Add your user prompt template here.
Use {text} as a placeholder for the document content.

Example:
Extract structured information from the following legal document:

{text}
"""

### OpenAI Request Functions

In [ ]:
def compose_openai_request(text_content: str) -> List[Dict[str, str]]:
    """
    Compose OpenAI API request messages from global prompt variables.
    
    Args:
        text_content: The document text content
        
    Returns:
        List of message dictionaries for OpenAI API
    """
    user_prompt = USER_PROMPT_TEMPLATE.format(text=text_content)
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]
    
    return messages


def call_openai_api(messages: List[Dict[str, str]], model: str = OPENAI_MODEL) -> Dict[str, Any]:
    """
    Make request to OpenAI API and parse JSON results.
    
    Args:
        messages: List of message dictionaries
        model: OpenAI model name
        
    Returns:
        Parsed JSON response from OpenAI
    """
    try:
        response = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            response_format={"type": "json_object"},
            temperature=0.0
        )
        
        content = response.choices[0].message.content
        parsed_json = json.loads(content)
        
        return parsed_json
    
    except Exception as e:
        raise RuntimeError(f"OpenAI API call failed: {e}")


def extract_information(text_content: str) -> Dict[str, Any]:
    """
    Extract structured information from text using OpenAI API.
    
    Args:
        text_content: The document text content
        
    Returns:
        Extracted JSON structure
    """
    messages = compose_openai_request(text_content)
    result = call_openai_api(messages)
    return result

## 4. BigQuery Functions

In [ ]:
def push_to_bigquery(
    document_id: str, 
    json_structure: Dict[str, Any],
    dataset_id: str = BQ_DATASET,
    table_id: str = BQ_TABLE
) -> None:
    """
    Push extracted JSON structure to BigQuery.
    
    Args:
        document_id: The document identifier
        json_structure: The extracted JSON structure
        dataset_id: BigQuery dataset ID
        table_id: BigQuery table ID
        
    Table Schema:
        - document_id: STRING
        - json_structure: STRING (JSON stored as string)
        - inserted_at: TIMESTAMP (auto-populated)
    """
    table_ref = f"{bq_client.project}.{dataset_id}.{table_id}"
    
    row = {
        "document_id": document_id,
        "json_structure": json.dumps(json_structure, ensure_ascii=False),
        "inserted_at": datetime.utcnow().isoformat()
    }
    
    errors = bq_client.insert_rows_json(table_ref, [row])
    
    if errors:
        raise RuntimeError(f"BigQuery insert failed: {errors}")
    
    print(f"✓ Pushed to BigQuery: {document_id}")

### Create BigQuery Table (Run Once)

In [ ]:
def create_bigquery_table(dataset_id: str = BQ_DATASET, table_id: str = BQ_TABLE) -> None:
    """
    Create BigQuery table with required schema if it doesn't exist.
    """
    table_ref = f"{bq_client.project}.{dataset_id}.{table_id}"
    
    schema = [
        bigquery.SchemaField("document_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("json_structure", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("inserted_at", "TIMESTAMP", mode="REQUIRED"),
    ]
    
    table = bigquery.Table(table_ref, schema=schema)
    
    try:
        table = bq_client.create_table(table)
        print(f"✓ Created table: {table_ref}")
    except Exception as e:
        if "Already Exists" in str(e):
            print(f"✓ Table already exists: {table_ref}")
        else:
            raise

# create_bigquery_table()

## 5. Complete Pipeline Functions

In [ ]:
def process_single_document(document_id: str, bucket_name: str) -> Dict[str, Any]:
    """
    Process a single document: download → parse → extract → upload.
    
    Args:
        document_id: The document ID
        bucket_name: GCS bucket name
        
    Returns:
        Dictionary with processing results and metadata
    """
    result = {
        "document_id": document_id,
        "status": "pending",
        "error": None
    }
    
    try:
        rtf_path = download_rtf_from_gcs(document_id, bucket_name)
        
        txt_path = TEMP_DIR / f"{document_id}.txt"
        text_content = parse_rtf_to_txt(rtf_path, txt_path)
        
        json_structure = extract_information(text_content)
        
        push_to_bigquery(document_id, json_structure)
        
        rtf_path.unlink()
        txt_path.unlink()
        
        result["status"] = "success"
        result["chars_processed"] = len(text_content)
        
    except Exception as e:
        result["status"] = "failed"
        result["error"] = str(e)
        print(f"✗ Error processing {document_id}: {e}")
    
    return result


def process_batch(
    document_ids: List[str], 
    bucket_name: str,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Process a batch of documents.
    
    Args:
        document_ids: List of document IDs to process
        bucket_name: GCS bucket name
        verbose: Print progress messages
        
    Returns:
        DataFrame with processing results for each document
    """
    results = []
    
    for idx, doc_id in enumerate(document_ids, 1):
        if verbose:
            print(f"\n[{idx}/{len(document_ids)}] Processing: {doc_id}")
            print("-" * 60)
        
        result = process_single_document(doc_id, bucket_name)
        results.append(result)
        
        if verbose:
            status_symbol = "✓" if result["status"] == "success" else "✗"
            print(f"{status_symbol} Status: {result['status']}")
    
    return pd.DataFrame(results)

## 6. Main Execution Pipeline

### Load and Filter Documents

In [ ]:
df_documents = pd.read_csv("documents.csv")

print(f"Total documents loaded: {len(df_documents)}")
print(f"\nDataFrame columns: {df_documents.columns.tolist()}")
print(f"\nFirst few rows:")
df_documents.head()

In [ ]:
# TODO: Add your filtering logic here
# Example filtering conditions:
# df_filtered = df_documents[df_documents['status'] == 'pending']
# df_filtered = df_documents[df_documents['year'] >= 2020]

df_filtered = df_documents.copy()

print(f"Documents after filtering: {len(df_filtered)}")

### Check Available Files in GCS

In [ ]:
def list_available_documents_in_gcs(bucket_name: str) -> List[str]:
    """
    List all RTF files available in GCS bucket.
    
    Returns:
        List of document IDs (filenames without .rtf extension)
    """
    bucket = storage_client.bucket(bucket_name)
    blobs = bucket.list_blobs()
    
    doc_ids = []
    for blob in blobs:
        if blob.name.endswith(".rtf"):
            doc_id = blob.name.replace(".rtf", "")
            doc_ids.append(doc_id)
    
    return doc_ids


available_gcs_docs = list_available_documents_in_gcs(GCS_BUCKET_NAME)
print(f"Available RTF files in GCS: {len(available_gcs_docs)}")

if "id" in df_filtered.columns:
    df_filtered["available_in_gcs"] = df_filtered["id"].isin(available_gcs_docs)
    df_to_process = df_filtered[df_filtered["available_in_gcs"] == True].copy()
    
    print(f"Documents to process (after GCS merge): {len(df_to_process)}")
    print(f"Documents missing in GCS: {len(df_filtered) - len(df_to_process)}")
else:
    print("Warning: 'id' column not found in DataFrame. Adjust merge logic.")
    df_to_process = df_filtered.copy()

### Batch Processing

In [ ]:
def process_documents_in_batches(
    df: pd.DataFrame,
    bucket_name: str,
    batch_size: int = BATCH_SIZE,
    id_column: str = "id"
) -> pd.DataFrame:
    """
    Process all documents in batches.
    
    Args:
        df: DataFrame with document information
        bucket_name: GCS bucket name
        batch_size: Number of documents to process per batch
        id_column: Column name containing document IDs
        
    Returns:
        DataFrame with all processing results
    """
    if id_column not in df.columns:
        raise ValueError(f"Column '{id_column}' not found in DataFrame")
    
    document_ids = df[id_column].tolist()
    total_docs = len(document_ids)
    
    print(f"{'='*60}")
    print(f"Starting batch processing")
    print(f"Total documents: {total_docs}")
    print(f"Batch size: {batch_size}")
    print(f"Total batches: {(total_docs + batch_size - 1) // batch_size}")
    print(f"{'='*60}\n")
    
    all_results = []
    
    for batch_num in range(0, total_docs, batch_size):
        batch_ids = document_ids[batch_num:batch_num + batch_size]
        batch_label = f"Batch {batch_num // batch_size + 1}"
        
        print(f"\n{'#'*60}")
        print(f"{batch_label}: Processing {len(batch_ids)} documents")
        print(f"{'#'*60}")
        
        batch_results = process_batch(batch_ids, bucket_name)
        all_results.append(batch_results)
    
    final_results = pd.concat(all_results, ignore_index=True)
    
    print(f"\n{'='*60}")
    print(f"PROCESSING COMPLETE")
    print(f"{'='*60}")
    print(f"Total processed: {len(final_results)}")
    print(f"Successful: {(final_results['status'] == 'success').sum()}")
    print(f"Failed: {(final_results['status'] == 'failed').sum()}")
    print(f"{'='*60}\n")
    
    return final_results

### Execute Processing

In [ ]:
results_df = process_documents_in_batches(
    df=df_to_process,
    bucket_name=GCS_BUCKET_NAME,
    batch_size=BATCH_SIZE,
    id_column="id"
)

results_df.head()

### View Results Summary

In [ ]:
print("Processing Summary:")
print(f"Total: {len(results_df)}")
print(f"\nStatus breakdown:")
print(results_df["status"].value_counts())

failed_docs = results_df[results_df["status"] == "failed"]
if len(failed_docs) > 0:
    print(f"\nFailed documents:")
    print(failed_docs[["document_id", "error"]])

### Save Results Log

In [ ]:
results_log_path = f"processing_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
results_df.to_csv(results_log_path, index=False)

print(f"✓ Results saved to: {results_log_path}")